# utilities

> This notebook provides utilities used in measuring the ICL in images of galaxy clusters.

In [ ]:
# | default_exp utilities

In [ ]:
# | export
import numpy as np
from astropy import units as u
from astropy.cosmology import FlatLambdaCDM
from astropy.nddata import CCDData, Cutout2D

from nicl import ezgal

In [ ]:
# | export


def calc_kcorr(z, filter):
    """Calculate the kcorrection from restframe B-band to an observation in `filter`.

    To convert the conventional B-band surface brightness limit to the observed band we
    use [ezgal](https://github.com/cmancone/easyGalaxy). As ezgal does not work correctly
    when installed via pip, and needs some data files and tweaks, we include a copy with nicl.

    """
    model = ezgal.model("bc03_ssp_z_0.02_chab.model")
    model.set_cosmology(Om=0.3, Ol=0.7, h=0.70, w=-1)
    zf = 3  # Redshift of formation, as in Furnell+21
    rest_b = model.get_absolute_mags(zf=zf, filters="B.dat", zs=z)
    obs_i = model.get_observed_absolute_mags(zf=zf, filters=filter, zs=z)
    kcorr = obs_i - rest_b
    return kcorr * u.mag


def calc_sb_threshold(z, filter, b_band_sb_threshold=25 * u.ABmag):
    k_corr = calc_kcorr(z, filter)
    dimming = 2.5 * np.log10((1 + z) ** 4) * u.mag
    return b_band_sb_threshold + k_corr + dimming


def sb_to_adu(sb, pix_scale, zp=27 * u.ABmag):
    sb_to_mag = sb - 2.5 * np.log10(pix_scale.to_value(u.arcsec) ** 2) * u.mag
    counts = 10 ** (-(sb_to_mag - zp).value / 2.5)
    return counts

In [ ]:
# | export


def get_pixel_scale(img):
    """Calculate the average pixel scale of the supplied image."""
    return (
        np.mean(
            [s.to_value("arcsec") for s in img.wcs.celestial.proj_plane_pixel_scales()]
        )
        * u.arcsec
    )


def get_img_centre_pixel(img):
    return (np.array(img.shape) - 1) / 2


def get_img_centre_world(img):
    """Calculate the world coordinates at the centre of the supplied image."""
    centre_pixel = get_img_centre_pixel(img)
    return img.wcs.pixel_to_world(*centre_pixel)


def distance_from_coord(shape, coord):
    """Calculate the distance from pixel `coord` to each pixel in an image with given `shape`."""
    ij = np.indices(shape)[::-1]
    coord = np.asarray(coord)[:, None, None]
    r = np.sqrt(((ij - coord) ** 2).sum(axis=0))
    return r

In [ ]:
# | export


def physical_to_angular(r_physical, z):
    """Convert a physical distance in the plane of the sky to an angle, for a given redshift `z`."""
    cosmo = FlatLambdaCDM(H0=70, Om0=0.3)
    deg_per_Mpc = 1 / cosmo.kpc_proper_per_arcmin(z).to(u.Mpc / u.deg)
    r_angular = r_physical * deg_per_Mpc
    return r_angular.to(u.deg)

In [ ]:
# | export


def pix2arcmin(
    axis: str,  # the plot axis, 'x' or 'y'
    img: CCDData,  # the image previously displayed with imshow
):
    """Return closures for transforming between pixel coordinates and arcmin offsets.

    These are for use when adding secondary axes to a matplotlib plot created with imshow, e.g.
    `ax.secondary_xaxis('bottom', functions=pix2arcmin('x', img))`.

    The pixel coordinates correspond to the use of `plt.imshow` without specifying an extent.
    Offsets are from the centre of the image.
    """
    idx = 0 if axis == "y" else 1
    pixel_size = get_pixel_scale(img)
    centre = get_img_centre_pixel(img)

    def forward(pix):
        return ((pix - centre[axis]) * pixel_size).to_value(u.arcmin)

    def inverse(arcmin):
        return ((arcmin * u.arcmin / pixel_size) + centre[idx]).to_value()

    return forward, inverse


def pix2Mpc(axis, img, z):
    """Return closures for transforming between pixel coordinates and Mpc offsets.

    These are for use when adding secondary axes to a matplotlib plot created with `plt.imshow`, e.g.
    `ax.secondary_xaxis('bottom', functions=pix2Mpc('x', img, z))`.

    The pixel coordinates correspond to the use of `plt.imshow` without specifying an extent.
    Offsets are from the centre of the image.
    """
    p2a, a2p = pix2arcmin(axis, img)
    arcmin_per_Mpc = physical_to_angular(1 * u.Mpc, z).to_value(u.arcmin)

    def forward(pix):
        return p2a(pix) / arcmin_per_Mpc

    def inverse(Mpc):
        return a2p(Mpc * arcmin_per_Mpc)

    return forward, inverse

In [ ]:
# | export


def get_cutout(image, size, position=None, mask=None, wcs=None):
    if position is None:
        position = get_img_centre_pixel(image).astype(int)
    if wcs is None:
        wcs = image.wcs
    cutout = Cutout2D(image, position, size, wcs=wcs, mode="partial").data
    if mask is not None:
        mask = Cutout2D(mask, position, size, wcs=wcs, mode="partial").data
        return cutout, mask
    else:
        return cutout

In [ ]:
# | hide
import nbdev

nbdev.nbdev_export()